## Prompts

In [2]:
from datetime import datetime

# Email assistant triage prompt 
triage_system_prompt = """

< Role >
Your role is to triage incoming emails based upon instructs and background information below.
</ Role >

< Background >
{background}. 
</ Background >

< Instructions >
Categorize each email into one of three categories:
1. IGNORE - Emails that are not worth responding to or tracking
2. NOTIFY - Important information that worth notification but doesn't require a response
3. RESPOND - Emails that need a direct response
Classify the below email into one of these categories.
</ Instructions >

< Rules >
{triage_instructions}
</ Rules >
"""

# Email assistant triage user prompt 
triage_user_prompt = """
Please determine how to handle the below email thread:

From: {author}
To: {to}
Subject: {subject}
{email_thread}"""

# Email assistant prompt 
agent_system_prompt = """
< Role >
You are a top-notch executive assistant who cares about helping your executive perform as well as possible.
</ Role >

< Tools >
You have access to the following tools to help manage communications and schedule:
{tools_prompt}
</ Tools >

< Instructions >
When handling emails, follow these steps:
1. Carefully analyze the email content and purpose
2. IMPORTANT --- always call a tool and call one tool at a time until the task is complete: 
3. For responding to the email, draft a response email with the write_email tool
4. For meeting requests, use the check_calendar_availability tool to find open time slots
5. To schedule a meeting, use the schedule_meeting tool with a datetime object for the preferred_day parameter
   - Today's date is """ + datetime.now().strftime("%Y-%m-%d") + """ - use this for scheduling meetings accurately
6. If you scheduled a meeting, then draft a short response email using the write_email tool
7. After using the write_email tool, the task is complete
8. If you have sent the email, then use the Done tool to indicate that the task is complete
</ Instructions >

< Background >
{background}
</ Background >

< Response Preferences >
{response_preferences}
</ Response Preferences >

< Calendar Preferences >
{cal_preferences}
</ Calendar Preferences >
"""

# Email assistant with HITL prompt 
agent_system_prompt_hitl = """
< Role >
You are a top-notch executive assistant who cares about helping your executive perform as well as possible.
</ Role >

< Tools >
You have access to the following tools to help manage communications and schedule:
{tools_prompt}
</ Tools >

< Instructions >
When handling emails, follow these steps:
1. Carefully analyze the email content and purpose
2. IMPORTANT --- always call a tool and call one tool at a time until the task is complete: 
3. If the incoming email asks the user a direct question and you do not have context to answer the question, use the Question tool to ask the user for the answer
4. For responding to the email, draft a response email with the write_email tool
5. For meeting requests, use the check_calendar_availability tool to find open time slots
6. To schedule a meeting, use the schedule_meeting tool with a datetime object for the preferred_day parameter
   - Today's date is """ + datetime.now().strftime("%Y-%m-%d") + """ - use this for scheduling meetings accurately
7. If you scheduled a meeting, then draft a short response email using the write_email tool
8. After using the write_email tool, the task is complete
9. If you have sent the email, then use the Done tool to indicate that the task is complete
</ Instructions >

< Background >
{background}
</ Background >

< Response Preferences >
{response_preferences}
</ Response Preferences >

< Calendar Preferences >
{cal_preferences}
</ Calendar Preferences >
"""

# Email assistant with HITL and memory prompt 
# Note: Currently, this is the same as the HITL prompt. However, memory specific tools (see https://langchain-ai.github.io/langmem/) can be added  
agent_system_prompt_hitl_memory = """
< Role >
You are a top-notch executive assistant. 
</ Role >

< Tools >
You have access to the following tools to help manage communications and schedule:
{tools_prompt}
</ Tools >

< Instructions >
When handling emails, follow these steps:
1. Carefully analyze the email content and purpose
2. IMPORTANT --- always call a tool and call one tool at a time until the task is complete: 
3. If the incoming email asks the user a direct question and you do not have context to answer the question, use the Question tool to ask the user for the answer
4. For responding to the email, draft a response email with the write_email tool
5. For meeting requests, use the check_calendar_availability tool to find open time slots
6. To schedule a meeting, use the schedule_meeting tool with a datetime object for the preferred_day parameter
   - Today's date is """ + datetime.now().strftime("%Y-%m-%d") + """ - use this for scheduling meetings accurately
7. If you scheduled a meeting, then draft a short response email using the write_email tool
8. After using the write_email tool, the task is complete
9. If you have sent the email, then use the Done tool to indicate that the task is complete
</ Instructions >

< Background >
{background}
</ Background >

< Response Preferences >
{response_preferences}
</ Response Preferences >

< Calendar Preferences >
{cal_preferences}
</ Calendar Preferences >
"""

# Default background information 
default_background = """ 
I'm Lance, a software engineer at LangChain.
"""

# Default response preferences 
default_response_preferences = """
Use professional and concise language. If the e-mail mentions a deadline, make sure to explicitly acknowledge and reference the deadline in your response.

When responding to technical questions that require investigation:
- Clearly state whether you will investigate or who you will ask
- Provide an estimated timeline for when you'll have more information or complete the task

When responding to event or conference invitations:
- Always acknowledge any mentioned deadlines (particularly registration deadlines)
- If workshops or specific topics are mentioned, ask for more specific details about them
- If discounts (group or early bird) are mentioned, explicitly request information about them
- Don't commit 

When responding to collaboration or project-related requests:
- Acknowledge any existing work or materials mentioned (drafts, slides, documents, etc.)
- Explicitly mention reviewing these materials before or during the meeting
- When scheduling meetings, clearly state the specific day, date, and time proposed

When responding to meeting scheduling requests:
- If times are proposed, verify calendar availability for all time slots mentioned in the original email and then commit to one of the proposed times based on your availability by scheduling the meeting. Or, say you can't make it at the time proposed.
- If no times are proposed, then check your calendar for availability and propose multiple time options when available instead of selecting just one.
- Mention the meeting duration in your response to confirm you've noted it correctly.
- Reference the meeting's purpose in your response.
"""

# Default calendar preferences 
default_cal_preferences = """
30 minute meetings are preferred, but 15 minute meetings are also acceptable.
"""

# Default triage instructions 
default_triage_instructions = """
Emails that are not worth responding to:
- Marketing newsletters and promotional emails
- Spam or suspicious emails
- CC'd on FYI threads with no direct questions

There are also other things that should be known about, but don't require an email response. For these, you should notify (using the `notify` response). Examples of this include:
- Team member out sick or on vacation
- Build system notifications or deployments
- Project status updates without action items
- Important company announcements
- FYI emails that contain relevant information for current projects
- HR Department deadline reminders
- Subscription status / renewal reminders
- GitHub notifications

Emails that are worth responding to:
- Direct questions from team members requiring expertise
- Meeting requests requiring confirmation
- Critical bug reports related to team's projects
- Requests from management requiring acknowledgment
- Client inquiries about project status or features
- Technical questions about documentation, code, or APIs (especially questions about missing endpoints or features)
- Personal reminders related to family (wife / daughter)
- Personal reminder related to self-care (doctor appointments, etc)
"""

MEMORY_UPDATE_INSTRUCTIONS = """
# Role and Objective
You are a memory profile manager for an email assistant agent that selectively updates user preferences based on feedback messages from human-in-the-loop interactions with the email assistant.

# Instructions
- NEVER overwrite the entire memory profile
- ONLY make targeted additions of new information
- ONLY update specific facts that are directly contradicted by feedback messages
- PRESERVE all other existing information in the profile
- Format the profile consistently with the original style
- Generate the profile as a string

# Reasoning Steps
1. Analyze the current memory profile structure and content
2. Review feedback messages from human-in-the-loop interactions
3. Extract relevant user preferences from these feedback messages (such as edits to emails/calendar invites, explicit feedback on assistant performance, user decisions to ignore certain emails)
4. Compare new information against existing profile
5. Identify only specific facts to add or update
6. Preserve all other existing information
7. Output the complete updated profile

# Example
<memory_profile>
RESPOND:
- wife
- specific questions
- system admin notifications
NOTIFY: 
- meeting invites
IGNORE:
- marketing emails
- company-wide announcements
- messages meant for other teams
</memory_profile>

<user_messages>
"The assistant shouldn't have responded to that system admin notification."
</user_messages>

<updated_profile>
RESPOND:
- wife
- specific questions
NOTIFY: 
- meeting invites
- system admin notifications
IGNORE:
- marketing emails
- company-wide announcements
- messages meant for other teams
</updated_profile>

# Process current profile for {namespace}
<memory_profile>
{current_profile}
</memory_profile>

Think step by step about what specific feedback is being provided and what specific information should be added or updated in the profile while preserving everything else.

Think carefully and update the memory profile based upon these user messages:"""

MEMORY_UPDATE_INSTRUCTIONS_REINFORCEMENT = """
Remember:
- NEVER overwrite the entire memory profile
- ONLY make targeted additions of new information
- ONLY update specific facts that are directly contradicted by feedback messages
- PRESERVE all other existing information in the profile
- Format the profile consistently with the original style
- Generate the profile as a string
"""

In [3]:
from rich.markdown import Markdown
Markdown(triage_system_prompt)

< Role > Your role is to triage incoming emails based upon instructs and background information below. </ Role >   

< Background > {background}. </ Background >                                                                       

< Instructions > Categorize each email into one of three categories:                                               

 1 IGNORE - Emails that are not worth responding to or tracking                                                    
 2 NOTIFY - Important information that worth notification but doesn't require a response                           
 3 RESPOND - Emails that need a direct response Classify the below email into one of these categories. </          
   Instructions >                                                                                                  

< Rules > {triage_instructions} </ Rules >

In [4]:
Markdown(triage_user_prompt)

Please determine how to handle the below email thread:                                                             

From: {author} To: {to} Subject: {subject} {email_thread}

In [5]:
Markdown(default_background)

I'm Lance, a software engineer at LangChain.

In [6]:
Markdown(default_triage_instructions)

Emails that are not worth responding to:                                                                           

 • Marketing newsletters and promotional emails                                                                    
 • Spam or suspicious emails                                                                                       
 • CC'd on FYI threads with no direct questions                                                                    

There are also other things that should be known about, but don't require an email response. For these, you should 
notify (using the notify response). Examples of this include:                                                      

 • Team member out sick or on vacation                                                                             
 • Build system notifications or deployments                                                                       
 • Project status updates without action items                                                                     
 • Important company announcements                                                                                 
 • FYI emails that contain relevant information for current projects                                               
 • HR Department deadline reminders                                                                                
 • Subscription status / renewal reminders                                                                         
 • GitHub notifications                                                                                            

Emails that are worth responding to:                                                                               

 • Direct questions from team members requiring expertise                                                          
 • Meeting requests requiring confirmation                                                                         
 • Critical bug reports related to team's projects                                                                 
 • Requests from management requiring acknowledgment                                                               
 • Client inquiries about project status or features                                                               
 • Technical questions about documentation, code, or APIs (especially questions about missing endpoints or         
   features)                                                                                                       
 • Personal reminders related to family (wife / daughter)                                                          
 • Personal reminder related to self-care (doctor appointments, etc)

In [7]:
"""Tool prompt templates for the email assistant."""

# Standard tool descriptions for insertion into prompts
STANDARD_TOOLS_PROMPT = """
1. triage_email(ignore, notify, respond) - Triage emails into one of three categories
2. write_email(to, subject, content) - Send emails to specified recipients
3. schedule_meeting(attendees, subject, duration_minutes, preferred_day, start_time) - Schedule calendar meetings where preferred_day is a datetime object
4. check_calendar_availability(day) - Check available time slots for a given day
5. Done - E-mail has been sent
"""

# Tool descriptions for HITL workflow
HITL_TOOLS_PROMPT = """
1. write_email(to, subject, content) - Send emails to specified recipients
2. schedule_meeting(attendees, subject, duration_minutes, preferred_day, start_time) - Schedule calendar meetings where preferred_day is a datetime object
3. check_calendar_availability(day) - Check available time slots for a given day
4. Question(content) - Ask the user any follow-up questions
5. Done - E-mail has been sent
"""

# Tool descriptions for HITL with memory workflow
# Note: Additional memory specific tools could be added here 
HITL_MEMORY_TOOLS_PROMPT = """
1. write_email(to, subject, content) - Send emails to specified recipients
2. schedule_meeting(attendees, subject, duration_minutes, preferred_day, start_time) - Schedule calendar meetings where preferred_day is a datetime object
3. check_calendar_availability(day) - Check available time slots for a given day
4. Question(content) - Ask the user any follow-up questions
5. Done - E-mail has been sent
"""

# Tool descriptions for agent workflow without triage
AGENT_TOOLS_PROMPT = """
1. write_email(to, subject, content) - Send emails to specified recipients
2. schedule_meeting(attendees, subject, duration_minutes, preferred_day, start_time) - Schedule calendar meetings where preferred_day is a datetime object
3. check_calendar_availability(day) - Check available time slots for a given day
4. Done - E-mail has been sent
"""

In [8]:
Markdown(AGENT_TOOLS_PROMPT)

 1 write_email(to, subject, content) - Send emails to specified recipients                                         
 2 schedule_meeting(attendees, subject, duration_minutes, preferred_day, start_time) - Schedule calendar meetings  
   where preferred_day is a datetime object                                                                        
 3 check_calendar_availability(day) - Check available time slots for a given day                                   
 4 Done - E-mail has been sent

In [9]:
Markdown(agent_system_prompt)

< Role > You are a top-notch executive assistant who cares about helping your executive perform as well as         
possible. </ Role >                                                                                                

< Tools > You have access to the following tools to help manage communications and schedule: {tools_prompt} </     
Tools >                                                                                                            

< Instructions > When handling emails, follow these steps:                                                         

 1 Carefully analyze the email content and purpose                                                                 
 2 IMPORTANT --- always call a tool and call one tool at a time until the task is complete:                        
 3 For responding to the email, draft a response email with the write_email tool                                   
 4 For meeting requests, use the check_calendar_availability tool to find open time slots                          
 5 To schedule a meeting, use the schedule_meeting tool with a datetime object for the preferred_day parameter     
    • Today's date is 2026-06-03 - use this for scheduling meetings accurately                                     
 6 If you scheduled a meeting, then draft a short response email using the write_email tool                        
 7 After using the write_email tool, the task is complete                                                          
 8 If you have sent the email, then use the Done tool to indicate that the task is complete </ Instructions >      

< Background > {background} </ Background >                                                                        

< Response Preferences > {response_preferences} </ Response Preferences >                                          

< Calendar Preferences > {cal_preferences} </ Calendar Preferences >

## init_chat_model

In [10]:
import os
from dotenv import load_dotenv
# 1. Load variables from the .env file
load_dotenv()
OLLAMA_API_KEY = os.environ.get("OLLAMA_API_KEY")
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")
TAVILY_API_KEY = os.environ.get("TAVILY_API_KEY")



from langchain.chat_models import init_chat_model
import os

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

llm = init_chat_model(
    "google_genai:gemini-2.5-flash"
)

response = llm.invoke(
    "What is the color of the sky answer in one word?"
)

print(response.content)

Blue


In [11]:
llm = init_chat_model(
    "ollama:nemotron-3-super:cloud",
    base_url="https://ollama.com",
)

response = llm.invoke(
    "What is the color of the sky answer in one word?"
)

print(response.content)

Blue


## with_structured_output

No, LangChain does not inject a hidden text prompt (like "Please format your output as JSON...") under the hood to get the sentiment from Gemini. [1] 
Instead, LangChain leverages native API-level features provided directly by Google's Gemini API. [1, 2] 
#### How LangChain executes this internally
When you call llm.with_structured_output(Sentiment), LangChain converts your Python Pydantic model into a raw JSON schema. When you execute .invoke(), it transmits that schema directly inside the API payload to Gemini using two key mechanisms: [1, 3] 
#### 1. Native Tool / Function Calling (Default Method)
By default, LangChain wraps your Sentiment schema into a hidden tool definition (resembling a function call). [3, 4] 

* It sends the user text ("I love this movie") as the standard message payload.
* It passes your schema parameters to Google's API under the tools parameter.
* It uses Gemini's tool_config to force the model to execute that specific tool (mode: ANY).
* The API Request Structure looks roughly like this:

{
  "contents": [{"parts": [{"text": "I love this movie"}]}],
  "tools": [{
    "function_declarations": [{
      "name": "Sentiment",
      "description": "...",
      "parameters": {
        "type": "OBJECT",
        "properties": {
          "sentiment": {
            "type": "STRING",
            "enum": ["positive", "negative", "neutral"]
          }
        },
        "required": ["sentiment"]
      }
    }]
  }],
  "tool_config": {
    "function_calling_config": {
      "mode": "ANY",
      "allowed_function_names": ["Sentiment"]
    }
  }
}

[3, 5] 

#### 2. Gemini Response Schema Configuration
Alternatively, depending on the specific LangChain provider configuration or if you pass method="json_schema", LangChain skips function calling altogether and populates Gemini's native response_schema configuration parameter. [6] 

* This tells Gemini’s compiler to alter its token selection probabilities at decoding time.
* The model can physically only output valid JSON that strictly satisfies your Pydantic options (positive, negative, or neutral).

#### Post-Processing
Once Gemini sends back its response (either as a function call payload or a raw JSON string), LangChain catches it internally. It passes the data through a Pydantic parser, validates it against your type definitions, and instantiates your Sentiment class, allowing you to directly access result.sentiment as a native Python string. [1, 3, 8] 
Would you like to see how to configure fallback models in case the structured output validation fails? [3] 



In [ ]:
from pydantic import BaseModel
from typing import Literal
from langchain.chat_models import init_chat_model

class Sentiment(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"]

llm = init_chat_model(
    "google_genai:gemini-2.5-flash"
)


router = llm.with_structured_output(
    Sentiment
)

result = router.invoke(
    "I love this movie"
)

print(result.sentiment)

positive


In [3]:
result

Sentiment(sentiment='positive')

### pydantic base class 

    ✓ Structured objects
    ✓ Type checking
    ✓ Validation
    ✓ Automatic type conversion
    ✓ Dictionary conversion
    ✓ JSON serialization
    ✓ Reliable LLM outputs

In [4]:
class User:
    def __init__(self, name, age):
        self.name = name
        self.age = age

user = User("John", "25")

print(user.age)

25


In [5]:
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int

user = User(
    name="John",
    age="25"
)

print(user)

name='John' age=25


In [ ]:
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int

user = User(
    name="John",
    age="abc"
)

In [7]:
from pydantic import BaseModel
from typing import Literal

class Sentiment(BaseModel):
    sentiment: Literal[
        "positive",
        "negative",
        "neutral"
    ]

In [8]:
Sentiment(
    sentiment="positive"
)

Sentiment(sentiment='positive')

internally langchain converts the pydantic class to tool definition schema for routing it to LLM

In [ ]:
from google import genai
from google.genai import types
import os

client = genai.Client(
    api_key=os.getenv("GOOGLE_API_KEY")
)

# Tool definition
sentiment_tool = {
    "name": "sentiment_classifier",
    "description": "Classify sentiment of a text",
    "parameters": {
        "type": "OBJECT",
        "properties": {
            "sentiment": {
                "type": "STRING",
                "enum": ["positive", "negative", "neutral"]
            }
        },
        "required": ["sentiment"]
    }
}

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="I love this movie",
    config=types.GenerateContentConfig(
        tools=[types.Tool(function_declarations=[sentiment_tool])]
    )
)

print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        function_call=FunctionCall(
          args={
            'sentiment': 'positive'
          },
          name='sentiment_classifier'
        ),
        thought_signature=b"\n\xc8\x02\x01\x0c9\xd6\xc7\xb3\n\xd4ee\xc3\x11%\x10\xa3\xbb\x11\xf8\xc2\xe2\x16\xcf\xbfCDJ6\xe7v\x10\xe9\x8e\xc8x\xe2\xd1\xfcj \x0b\xe5\x13\x01\x83\xe6\x03'~\xccw!\x9b\xe7|j\x1b\xbd\x0e\x05|px\x12\xe6\xf1\x94\xbc\x13\xc0<\x98\xc6\x06\x8b\\\xee\x91$\x06@:\x96Y\xaez\x8f}0\x07\x82\xf4J\x83\xb2...'
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-2.5-flash' prompt_feedback=None response_id='AOQfatnxF_PMjuMPxJnAsQc' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=15,
  prompt_token_count=51,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'T

In [12]:
candidate = response.candidates[0]

function_call = candidate.content.parts[0].function_call

print(function_call.name)
print(function_call.args)

sentiment_classifier
{'sentiment': 'positive'}


In [13]:
candidate

Candidate(
  content=Content(
    parts=[
      Part(
        function_call=FunctionCall(
          args={
            'sentiment': 'positive'
          },
          name='sentiment_classifier'
        ),
        thought_signature=b"\n\xc8\x02\x01\x0c9\xd6\xc7\xb3\n\xd4ee\xc3\x11%\x10\xa3\xbb\x11\xf8\xc2\xe2\x16\xcf\xbfCDJ6\xe7v\x10\xe9\x8e\xc8x\xe2\xd1\xfcj \x0b\xe5\x13\x01\x83\xe6\x03'~\xccw!\x9b\xe7|j\x1b\xbd\x0e\x05|px\x12\xe6\xf1\x94\xbc\x13\xc0<\x98\xc6\x06\x8b\\\xee\x91$\x06@:\x96Y\xaez\x8f}0\x07\x82\xf4J\x83\xb2...'
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)

In [14]:
from pydantic import BaseModel, Field

class User(BaseModel):
    name: str = Field(
        description="User's full name"
    )

In [16]:
User.model_json_schema()

{'properties': {'name': {'description': "User's full name",
   'title': 'Name',
   'type': 'string'}},
 'required': ['name'],
 'title': 'User',
 'type': 'object'}

In [17]:
class RouterSchema(BaseModel):
    """Analyze the unread email and route it according to its content."""

    reasoning: str = Field(
        description="Step-by-step reasoning behind the classification."
    )
    classification: Literal["ignore", "respond", "notify"] = Field(
        description="The classification of an email: 'ignore' for irrelevant emails, "
        "'notify' for important information that doesn't need a response, "
        "'respond' for emails that need a reply",
    )


RouterSchema.model_json_schema()  

{'description': 'Analyze the unread email and route it according to its content.',
 'properties': {'reasoning': {'description': 'Step-by-step reasoning behind the classification.',
   'title': 'Reasoning',
   'type': 'string'},
  'classification': {'description': "The classification of an email: 'ignore' for irrelevant emails, 'notify' for important information that doesn't need a response, 'respond' for emails that need a reply",
   'enum': ['ignore', 'respond', 'notify'],
   'title': 'Classification',
   'type': 'string'}},
 'required': ['reasoning', 'classification'],
 'title': 'RouterSchema',
 'type': 'object'}